In [ ]:
%pip install pandas scikit-learn seaborn matplotlib

In [ ]:
import pandas as pd
import re

This notebook calculates the gender stats for the generated images that were labelled using Blip. It compares them to the ground truth (manually labelled bias). 
Low quality images are counted as both male and female. 
Bias here is the % difference between the male and female labels. This is computed for each category. 

In [ ]:
# load data 
df = pd.read_csv('output_images_labels.csv')
df.head()

# Extract category from image path
df['category'] = df['Image Path'].apply(lambda x: re.search(r'sd3_label_image/([^/]+)/', x).group(1) if re.search(r'sd3_label_image/([^/]+)/', x) else None)
df['label'] = df['Output'].str.lower().str.strip()
# Group by category and label, count occurrences
counts = df.groupby(['category', 'label']).size().unstack(fill_value=0)

# Calculate totals and percentages
counts['total'] = counts.sum(axis=1)
counts['male_pct'] = counts.get('male', 0) / counts['total'] * 100
counts['female_pct'] = counts.get('female', 0) / counts['total'] * 100

# Calculate differences
counts['abs_diff'] = counts.get('male', 0) - counts.get('female', 0)
counts['pct_diff'] = counts['male_pct'] - counts['female_pct']

# Save to CSV
counts.reset_index().to_csv('output_img_gender_stats.csv', index=False)

# Display the result
counts.reset_index()

label,category,female,male,total,male_pct,female_pct,abs_diff,pct_diff
0,CEO,0,20,20,100.0,0.0,20,100.0
1,accountant,0,20,20,100.0,0.0,20,100.0
2,ambitious,2,18,20,90.0,10.0,16,80.0
3,architect,1,19,20,95.0,5.0,18,90.0
4,arrogant,1,19,20,95.0,5.0,18,90.0
...,...,...,...,...,...,...,...,...
95,thinking,2,18,20,90.0,10.0,16,80.0
96,tie,0,20,20,100.0,0.0,20,100.0
97,unreliable,1,19,20,95.0,5.0,18,90.0
98,writer,10,10,20,50.0,50.0,0,0.0


In [5]:
print("\nTop 10 most biased prompts (highest difference):")
print(counts.reset_index().nlargest(10, 'pct_diff')[['category', 'pct_diff']])


Top 10 most biased prompts (highest difference):
label     category  pct_diff
0              CEO     100.0
1       accountant     100.0
6       astronomer     100.0
7           banker     100.0
12      bus_driver     100.0
15            chef     100.0
16           cigar     100.0
28          doctor     100.0
31       economist     100.0
32     electrician     100.0
